# Model B v2 — train the stronger depthwise CNN
**KavijaK · EuroSAT RGB · 96,538 parameters · trained from scratch**

Uses MobileNetV2-style expanded depthwise blocks, linear residual connections, batch normalization, SE channel attention and global average pooling. This is the final Model B implementation. Its architecture and training recipe differ from Model A; compare complete recipes when interpreting accuracy.

**Setup:** choose **Runtime → Change runtime type → T4 GPU**. Run cells in order; upload **EN3150_A03_ModelB_v2_Code.zip** when prompted. No GitHub login is needed. Training takes 60 epochs and saves checkpoints to Drive. Leave the final-test switch off while comparing validation results.

The completed AdamW run achieved **98.12% test accuracy**, **98.12% macro precision**, and **97.99% macro recall** on 4,050 images. Results are committed under `results/model_b/modelB_v2_adamw_01/`. This runnable notebook has no prefilled execution outputs. The optimizer comparison is complete: **SGD achieved 96.00%**, and **momentum SGD achieved 97.85%** test accuracy, also after 60 epochs. All three saved runs are in `results/model_b/`. No further mandatory Model B training is needed.


In [ ]:
import sys, subprocess, json, io, zipfile, hashlib, shutil
from pathlib import Path, PurePosixPath
import torch
from google.colab import files, drive
from IPython.display import display, Image, Markdown

assert torch.cuda.is_available(), "Choose a GPU runtime before continuing."
PROJECT_ROOT = Path("/content/model_b_v2/EN3150_A03")
CODE_ZIP = Path("/content/EN3150_A03_ModelB_v2_Code.zip")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)

if not (PROJECT_ROOT / "scripts/train_model_b_v2.py").is_file():
    uploaded = files.upload()
    names = [name for name in uploaded if name.endswith(".zip")]
    if len(names) != 1:
        raise ValueError("Upload exactly one Model B v2 code ZIP.")
    payload = uploaded[names[0]]
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if (path.is_absolute() or ".." in path.parts or chr(92) in info.filename
                or not path.parts or path.parts[0] != "EN3150_A03"
                or (info.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError(f"Unexpected archive entry: {info.filename}")
        archive.extractall(PROJECT_ROOT.parent)
    CODE_ZIP.write_bytes(payload)
    del uploaded, payload
assert (PROJECT_ROOT / "scripts/train_model_b_v2.py").is_file()
print("Source:", PROJECT_ROOT)
print("To switch code versions, start a fresh runtime and choose a new run ID.")


## 1. Dependencies, persistent storage and frozen dataset
This keeps Colab's existing PyTorch/CUDA installation. The source archive is saved to Drive alongside your experiments. Reuse the previously downloaded EuroSAT images if present; otherwise download the official RGB archive. The setup verifies the committed split without changing it.


In [ ]:
import importlib.util
missing = [package for module, package in {
    "numpy": "numpy", "PIL": "Pillow", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "yaml": "PyYAML", "torchvision": "torchvision"
}.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

def execute(command):
    process = subprocess.Popen([str(arg) for arg in command], cwd=PROJECT_ROOT,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError("Command failed; inspect the output above.")
    except BaseException:
        process.terminate()
        process.wait()
        raise

drive.mount("/content/drive")
RESULTS_ROOT = Path("/content/drive/MyDrive/EN3150_A03/model_b")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
if CODE_ZIP.exists():
    source_sha = hashlib.sha256(CODE_ZIP.read_bytes()).hexdigest()
    source_dir = RESULTS_ROOT.parent / "model_b_v2_source"
    source_dir.mkdir(exist_ok=True)
    saved_source = source_dir / f"{source_sha}.zip"
    if not saved_source.exists():
        shutil.copy2(CODE_ZIP, saved_source)
    print("Source archive saved:", saved_source)

previous_data = Path("/content/EN3150_A03/data/raw/EuroSAT_RGB")
DATA_ROOT = previous_data if previous_data.is_dir() else PROJECT_ROOT / "data/raw/EuroSAT_RGB"
execute([sys.executable, "scripts/prepare_model_b_data.py", "--image-root", DATA_ROOT,
         "--archive", DATA_ROOT.parent / "EuroSAT_RGB.zip"])


## 2. Run settings
**All three optimizer runs are complete.** This notebook is retained for reproducibility. Choose `OPTIMIZER = "adamw"`, `"sgd"` or `"sgd_momentum"` and a new run ID if repeating a run. Keep 60 epochs and batch size 128. The defaults describe the completed primary recipe; existing results cannot be overwritten silently. Training uses only train/validation data and selects the checkpoint with lowest validation cross-entropy.

For a disconnected run, upload the **same code ZIP**, keep the settings and run ID identical, set `RESUME = True`, then rerun setup and training. Completed epochs are retained. Use a new run ID whenever changing optimizer, architecture, seed or training settings. Resume checks source, dataset and runtime versions.

The assignment compares a chosen primary optimizer with SGD and momentum SGD. AdamW is the primary optimizer; a separate `"adam"` run is optional. To study attention, set `USE_ATTENTION = False` with a new run ID and the same recipe; this variant has 87,546 parameters. Compare validation results before evaluating the final test.


In [ ]:
RUN_ID = "modelB_v2_adamw_01"
OPTIMIZER = "adamw"  # adamw, adam, sgd, sgd_momentum
EPOCHS = 60
BATCH_SIZE = 128
SEED = 42
USE_ATTENTION = True
USE_AUGMENTATION = True
RESUME = False

RUN_DIR = RESULTS_ROOT / RUN_ID
command = [sys.executable, "scripts/train_model_b_v2.py", "--data-root", DATA_ROOT,
           "--results-dir", RESULTS_ROOT, "--run-id", RUN_ID, "--optimizer", OPTIMIZER,
           "--epochs", EPOCHS, "--batch-size", BATCH_SIZE, "--seed", SEED,
           "--num-workers", 2, "--cpu-threads", 2, "--device", "cuda"]
if not USE_ATTENTION:
    command += ["--no-attention"]
if not USE_AUGMENTATION:
    command += ["--no-augmentation"]
if RESUME:
    command += ["--resume"]
execute(command)


## 3. Review validation
Training accuracy/loss uses random augmentation and label smoothing; validation uses deterministic normalization and ordinary cross-entropy. The two losses are not directly comparable. The saved epoch minimizes validation loss, which may differ from the epoch with highest validation accuracy.


In [ ]:
summary = json.loads((RUN_DIR / "training_summary.json").read_text())
print(json.dumps(summary, indent=2))
display(Image(filename=str(RUN_DIR / "loss_curves.png")))


## 4. Final test — run after choosing your recipe
Set `EVALUATE_FINAL_TEST = True` only once the recipe is fixed from validation. The script loads the saved best checkpoint and produces accuracy, macro precision/recall, per-class scores and a confusion matrix. Existing test results are reused. Do not choose architectures or tune hyperparameters using test accuracy.


In [ ]:
EVALUATE_FINAL_TEST = False
if EVALUATE_FINAL_TEST:
    execute([sys.executable, "scripts/train_model_b_v2.py", "--data-root", DATA_ROOT,
             "--results-dir", RESULTS_ROOT, "--run-id", RUN_ID,
             "--device", "cuda", "--num-workers", 2, "--evaluate-only"])
    metrics = json.loads((RUN_DIR / "test_metrics.json").read_text())
    print({key: metrics[key] for key in ("accuracy", "macro_precision", "macro_recall")})
    display(Image(filename=str(RUN_DIR / "confusion_matrix.png")))
else:
    print("Final test skipped. Review validation first.")


## 5. Generate the report and download integration files
The export contains only this new run's metrics, curves, reports and configuration. Extract it in your repository; its paths are `results/model_b/<RUN_ID>/...`. Weights and recovery checkpoints remain on Drive. Teammate runs are unchanged. Only final Model B artifacts are exported.

Final Model B has more parameters and compute than Model A. Report the accuracy/compute tradeoff honestly. Comparisons change the complete training recipe; the attention ablation is needed to isolate attention's contribution.


In [ ]:
REPORT_DIR = RUN_DIR / "report"
execute([sys.executable, "scripts/report_model_b_v2.py", "--run-dir", RUN_DIR, "--output-dir", REPORT_DIR])
display(Markdown((REPORT_DIR / "experiment_report.md").read_text()))
archive_path = Path("/content") / f"{RUN_ID}_integration.zip"
artifacts = [path for path in RUN_DIR.iterdir()
             if path.is_file() and path.suffix.lower() in {".json", ".csv", ".png", ".md"}]
artifacts.append(REPORT_DIR / "experiment_report.md")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(artifacts):
        archive.write(path, Path("results/model_b") / RUN_ID / path.relative_to(RUN_DIR))
print("Checkpoints and weights remain at", RUN_DIR)
files.download(str(archive_path))


## Paper foundations
This is a custom small architecture inspired by published methods, not a claim of a new convolution method or reproduction of a paper's scores.

- [MobileNets: depthwise separable convolutions](https://arxiv.org/abs/1704.04861)
- [MobileNetV2: inverted residuals and linear bottlenecks](https://arxiv.org/abs/1801.04381)
- [Squeeze-and-Excitation Networks: channel attention](https://arxiv.org/abs/1709.01507)
- [MobileNetV3: hard sigmoid gating](https://arxiv.org/abs/1905.02244)
- [Network In Network: global average pooling](https://arxiv.org/abs/1312.4400)
- [Batch Normalization](https://arxiv.org/abs/1502.03167)
- [AdamW: decoupled weight decay](https://arxiv.org/abs/1711.05101)

The model combines expanded residual blocks, normalization, attention and global average pooling under the 100,000-parameter budget.
